### Lab 9

### 9.6.1 Support Vector Classifier

In [8]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import confusion_matrix

np.random.seed(1)

# 1. Generate data
X = np.random.randn(20, 2)
y = np.array([-1]*10 + [1]*10)
X[y == 1] += 1

print("High Cost Model (C=10)")
# Fit SVC
svm_fit_10 = SVC(kernel="linear", C=10.0)
svm_fit_10.fit(X, y)

print(f"Number of support vectors: {len(svm_fit_10.support_)}")
print(f"Indices of support vectors: {svm_fit_10.support_}")
print(f"Support vectors per class: {svm_fit_10.n_support_}\n")

print("Low Cost Model (C=0.1)")
svm_fit_01 = SVC(kernel="linear", C=0.1)
svm_fit_01.fit(X, y)

print(f"Number of support vectors: {len(svm_fit_01.support_)}")
print(f"Indices of support vectors: {svm_fit_01.support_}\n")

# 2. Cross-Validation GridSearchCV
print("Tuning Hyperparameters via 10-Fold CV")
tuned_parameters = [{"C": [0.001, 0.01, 0.1, 1, 5, 10, 100]}]

cv_strategy = KFold(n_splits=10, shuffle=True, random_state=1)
clf = GridSearchCV(SVC(kernel="linear"), tuned_parameters, cv=cv_strategy, scoring="accuracy")
clf.fit(X, y)

cv_results = pd.DataFrame(clf.cv_results_)[["param_C", "mean_test_score"]]
cv_results["mean_error"] = 1 - cv_results["mean_test_score"]
print(cv_results[['param_C', 'mean_error']].to_string(index=False))
print(f"\nBest parameter C found: {clf.best_params_['C']}\n")

# 3. Test best model
print("Evaluating on Test Data")
np.random.seed(2)
X_test = np.random.randn(20, 2)
y_test = np.random.choice([-1, 1], size=20, replace=True)
X_test[y_test == 1] += 1

# Prediction (C=1.0)
best_mod = clf.best_estimator_
y_pred_best = best_mod.predict(X_test)
print("Confusion Matrix for Best Model (C=1):")
print(confusion_matrix(y_test, y_pred_best, labels=[-1, 1]))

# Prediction с C=0.01
svm_fit_001 = SVC(kernel="linear", C=0.01).fit(X, y)
y_pred_001 = svm_fit_001.predict(X_test)
print("\nConfusion Matrix for Model with C=0.01:")
print(confusion_matrix(y_test, y_pred_001, labels=[-1, 1]))
print("\n")

# 4. Overfitting vs Stability
print("Linearly Separable Case")
X_sep = X.copy()
X_sep[y == 1] += 0.5

# Model №1: large C
svm_separable_hard = SVC(kernel="linear", C=1e5).fit(X_sep, y)
print("Hard Margin (C=1e5):")
print(f"Number of support vectors: {len(svm_separable_hard.support_)}")
print(f"Support vectors per class: {svm_separable_hard.n_support_}")

# Model №2: small C
svm_separable_soft = SVC(kernel="linear", C=1.0).fit(X_sep, y)
print("\nSoft Margin (C=1.0):")
print(f"Number of support vectors: {len(svm_separable_soft.support_)}")
print(f"Support vectors per class: {svm_separable_soft.n_support_}")


High Cost Model (C=10)
Number of support vectors: 6
Indices of support vectors: [ 0  4  9 13 15 16]
Support vectors per class: [3 3]

Low Cost Model (C=0.1)
Number of support vectors: 15
Indices of support vectors: [ 0  3  4  6  7  8  9 11 12 13 14 15 16 17 18]

Tuning Hyperparameters via 10-Fold CV
 param_C  mean_error
   0.001        0.25
   0.010        0.25
   0.100        0.15
   1.000        0.10
   5.000        0.10
  10.000        0.10
 100.000        0.10

Best parameter C found: 1

Evaluating on Test Data
Confusion Matrix for Best Model (C=1):
[[11  5]
 [ 1  3]]

Confusion Matrix for Model with C=0.01:
[[11  5]
 [ 1  3]]


Linearly Separable Case
Hard Margin (C=1e5):
Number of support vectors: 3
Support vectors per class: [2 1]

Soft Margin (C=1.0):
Number of support vectors: 6
Support vectors per class: [3 3]


conclusions:
- The Support Vector Classifier realised on Python is a bit defferent. The more tuning parameter C the darrower margin and less the number of support vectors. And the less tunining parameter the more number of support vectors
- Cross-Validation showed that, as parameter C grows the mean error decreases steadily and not sharp. The minimum is located when C = 100. But the same level of error can be reached with C = 1 (using the rule of one mean deviation maybe we should select 1).
- Confusion matrix (C=1.0): Precision = 0.68, Recall=0.91 -> The model is too bold. Makes a lot of predictions which is connected with TP, but precision is quite high, it means the model makes quite many mistakes connected with FP.
- Confusion matrix (C=0,01): the same as previous one

- The scikit-learn implementation of the Support Vector Classifier in Python uses an inverse regularization parameter $C$ compared to the textbook formulation. A larger $C$ imposes a heavier penalty on margin violations, leading to a narrower margin and fewer support vectors (e.g., 6 support vectors at $C=10$). Conversely, a smaller $C$ relaxes the penalty, resulting in a wider margin and a larger number of support vectors (e.g., 15 support vectors at $C=0.1$).
- Ten-fold cross-validation demonstrated that the mean classification error steadily decreases as $C$ increases, stabilizing at a minimum error rate of 0.10 for $C \ge 1$. Following the "one-standard-error rule" (or preferring the most parsimonious model), $C=1$ is selected as the optimal hyperparameter since it achieves the lowest error while maintaining a wider, more robust margin than higher values like $C=100$.
- The optimal model ($C=1$) achieved a baseline performance on the test set with a high Recall for the negative class ($11/12 \approx 0.91$) but a lower Precision ($11/16 \approx 0.68$), indicating a tendency to commit False Positive errors. Due to the small sample size of the test set ($n=20$), lowering the parameter drastically to $C=0.01$ yielded an identical confusion matrix, showing no immediate change in classification boundaries for these specific test instances
-  the linearly separable case, a very large parameter ($C=1e5$) forces a Hard Margin solution. This model fits the training observations perfectly (low bias) but relies on only 3 support vectors, making it highly susceptible to overfitting (high variance). Introducing a Soft Margin ($C=1.0$) allows minor training misclassifications but broadens the margin to include 6 support vectors. This effectively reduces model variance and enhances generalization on unseen data